In [1]:
import os

In [2]:
def get_file_list(data_folder):
    return [os.path.join(dirpath, file).replace(os.path.sep, '/') for dirpath, dirnames, files in os.walk(data_folder) for file in files]

In [3]:
data_folder = "UPAS DATA"
get_file_list(data_folder)

['UPAS DATA/20241222/PSP00006_LOG_2024-12-20T10_12_01UTC_BM0457M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00007_LOG_2024-12-20T09_52_23UTC_BM0234M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00019_LOG_2024-12-20T08_39_36UTC_BM1313M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00024_LOG_2024-12-20T08_41_06UTC_BM1314M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00027_LOG_2024-12-20T09_47_30UTC_BM0227M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00028_LOG_2024-12-20T10_02_23UTC_BM1069M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00029_LOG_2024-12-20T09_45_39UTC_BM1467M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00032_LOG_2024-12-20T10_10_14UTC_BM1296M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00036_LOG_2024-12-20T10_07_58UTC_BM1501M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00041_LOG_2024-12-20T10_00_58UTC_BM0580M_________DUMMY_____.txt',
 'UPAS DATA/20241222/PSP00043_LOG_2024-12-20T08_27_02UTC_BM0847M_________DUMMY_____.txt',
 'UPAS DAT

In [1]:
import os
import pandas as pd
import re
import traceback
from tqdm import tqdm
import datetime
pd.set_option('display.max_columns', None)

def get_file_list(data_folder):
    """Get a list of all files in the data folder and its subfolders."""
    return [os.path.join(dirpath, file).replace(os.path.sep, '/') 
            for dirpath, dirnames, files in os.walk(data_folder) 
            for file in files if file.endswith('.txt')]

def open_output_folder(folder_path):
    """Open the output folder in the system's file explorer"""
    import os
    import platform
    import subprocess
    
    folder_path = os.path.abspath(folder_path)
    
    if platform.system() == "Windows":
        # For Windows
        os.startfile(folder_path)
    elif platform.system() == "Darwin":
        # For macOS
        subprocess.Popen(["open", folder_path])
    else:
        # For Linux
        subprocess.Popen(["xdg-open", folder_path])

def extract_info_from_filename(file_path):
    """
    Extract mstudyid and filedate from the filename.
    
    Returns:
    - mstudyid: Participant ID (e.g., BM0457M)
    - filedate: File date in datetime format
    """
    filename = os.path.basename(file_path)
    
    # Extract mstudyid using regex
    # Pattern looks for section after UTC_ and before _DUMMY
    mstudyid_match = re.search(r'UTC_([A-Za-z0-9]+)_+DUMMY', filename)
    mstudyid = mstudyid_match.group(1) if mstudyid_match else "Unknown"
    
    # Extract filedate using regex
    # Pattern looks for date format like 2024-12-20T10_12_01
    filedate_str_match = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})', filename)
    filedate_str = filedate_str_match.group(1) if filedate_str_match else None
    
    # Convert filedate string to datetime object
    filedate = None
    if filedate_str:
        try:
            # Replace underscores with colons in the time part
            formatted_str = filedate_str.replace('_', ':')
            filedate = pd.to_datetime(formatted_str)
        except:
            pass
    
    return mstudyid, filedate_str, filedate

def parse_upas_file(file_path):
    """
    Parse UPAS data file into two dataframes with error handling.
    Returns:
    - properties_df: Contains metadata with "PARAMETER", "VALUE", "UNITS/NOTES" columns
    - data_df: Contains sample log data
    - error: Error message if any
    """
    # Extract mstudyid and filedate from filename
    mstudyid, filedate_str, filedate = extract_info_from_filename(file_path)
    
    # Try different encodings
    encodings = ['utf-8', 'latin1', 'cp1252', 'ISO-8859-1']
    lines = None
    
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                lines = f.readlines()
            break  # If successful, exit the loop
        except Exception as e:
            continue  # Try the next encoding
    
    if lines is None:
        return None, None, f"Failed to decode file with any of the attempted encodings"
        
    try:
        # Find the line index where "SAMPLE LOG" appears
        sample_log_index = None
        for i, line in enumerate(lines):
            if "SAMPLE LOG" in line:
                sample_log_index = i
                break
        
        if sample_log_index is None:
            return None, None, "Could not find 'SAMPLE LOG' marker in file"
        
        # Parse properties
        properties_data = []
        
        # Skip first line and read until SAMPLE LOG
        for line in lines[1:sample_log_index]:
            line = line.strip()
            # Skip empty lines and section headers (all capital letters with no commas)
            if not line or "," not in line or (line.isupper() and "," not in line):
                continue
            
            # Extract parts using regex to handle parentheses in comments properly
            parts = re.split(r',([^,(]*(?:\([^)]*\)[^,]*)*)', line, 1)
            
            if len(parts) >= 2:
                parameter = parts[0].strip()
                value = parts[1].strip()
                
                # Extract units/notes (everything after the second comma)
                units_notes = ""
                if len(parts) > 2 and parts[2]:
                    units_notes = parts[2].strip()
                    # Remove leading comma if present
                    if units_notes.startswith(','):
                        units_notes = units_notes[1:].strip()
                
                properties_data.append({
                    "FILEPATH": file_path,
                    "mstudyid": mstudyid,
                    "filedate": filedate,
                    "PARAMETER": parameter,
                    "VALUE": value,
                    "UNITS/NOTES": units_notes
                })
        
        # Create properties DataFrame
        properties_df = pd.DataFrame(properties_data)
        
        # Parse data
        # The header is 3 lines after SAMPLE LOG
        header_index = sample_log_index + 3
        if header_index >= len(lines):
            return properties_df, None, "File format error: header line not found after SAMPLE LOG"
        
        header = lines[header_index].strip().split(',')
        
        # Skip the units line
        data_start_index = header_index + 2
        
        # Process data lines
        data_content = []
        for line in lines[data_start_index:]:
            line = line.strip()
            if not line or line.startswith('-----'):
                continue  # Skip separator lines or empty lines
            
            values = line.split(',')
            if len(values) > 1:  # Ensure it's a data line
                # If there are fewer values than headers, pad with NaN
                if len(values) < len(header):
                    values.extend([''] * (len(header) - len(values)))
                # If there are more values than headers, truncate
                elif len(values) > len(header):
                    values = values[:len(header)]
                    
                data_content.append(values)
        
        # Create data DataFrame
        if data_content:
            data_df = pd.DataFrame(data_content, columns=header)
            # Add filepath and extracted info columns at the beginning
            data_df.insert(0, 'FILEPATH', file_path)
            data_df.insert(1, 'mstudyid', mstudyid)
            data_df.insert(2, 'filedate', filedate)
        else:
            # Create empty DataFrame with FILEPATH, mstudyid, and filedate as first columns
            columns = ['FILEPATH', 'mstudyid', 'filedate'] + header
            data_df = pd.DataFrame(columns=columns)
        
        return properties_df, data_df, None
        
    except Exception as e:
        error_detail = traceback.format_exc()
        return None, None, f"Error: {str(e)}\n{error_detail}"

def log_error(output_dir, error_message, file_path):
    """Log errors to a file"""
    os.makedirs(output_dir, exist_ok=True)
    error_log_path = os.path.join(output_dir, "error_log.txt")
    
    with open(error_log_path, 'a') as f:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{timestamp}] {os.path.basename(file_path)}: {error_message}\n")
        f.write("-" * 80 + "\n")

def process_upas_files(data_folder, output_dir="output"):
    """
    Process all UPAS files in the data folder and combine the results.
    Saves results to CSV files and returns the combined dataframes.
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get file list
    file_list = get_file_list(data_folder)
    total_files = len(file_list)
    
    # Prepare data containers
    all_properties = []
    all_data = []
    success_count = 0
    error_count = 0
    
    # Track original column order from first valid file
    original_columns = None
    
    # Process files with progress bar
    with tqdm(total=total_files, desc="Processing files", unit="file", 
             bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}") as pbar:
        
        for file_path in file_list:
            try:
                prop_df, data_df, error = parse_upas_file(file_path)
                
                if error:
                    error_count += 1
                    pbar.set_postfix_str(f"Success: {success_count}, Errors: {error_count}")
                    log_error(output_dir, f"Error processing {file_path}: {error}", file_path)
                else:
                    success_count += 1
                    if prop_df is not None:
                        all_properties.append(prop_df)
                    if data_df is not None:
                        # If this is the first valid dataframe, capture original column order
                        if original_columns is None and not data_df.empty:
                            # Get all columns except our added columns which we'll place at the beginning
                            original_data_columns = [col for col in data_df.columns 
                                                   if col not in ['FILEPATH', 'mstudyid', 'filedate']]
                            # Store the original order with our columns at the beginning
                            original_columns = ['FILEPATH', 'mstudyid', 'filedate'] + original_data_columns
                            
                        all_data.append(data_df)
                    pbar.set_postfix_str(f"Success: {success_count}, Errors: {error_count}")
            except Exception as e:
                error_count += 1
                error_detail = traceback.format_exc()
                pbar.set_postfix_str(f"Success: {success_count}, Errors: {error_count}")
                log_error(output_dir, f"Error processing {file_path}: {str(e)}\n{error_detail}", file_path)
            
            pbar.update(1)
    
    # Combine all dataframes
    if all_properties:
        combined_properties = pd.concat(all_properties, ignore_index=True)
        combined_properties.to_csv(os.path.join(output_dir, "combined_properties.csv"), index=False)
        print(f"Combined properties saved with {len(combined_properties)} rows")
    else:
        combined_properties = pd.DataFrame()
        print("No property data was successfully processed")
    
    if all_data:
        # Collect all unique columns from all dataframes
        all_columns = set()
        for df in all_data:
            all_columns.update(df.columns)
        
        # Make sure our important columns are first in the column list
        columns_to_use = ['FILEPATH', 'mstudyid', 'filedate']
        
        # If we have the original column order, use it
        if original_columns:
            # Add any original columns that aren't already in the list
            for col in original_columns:
                if col not in columns_to_use:
                    columns_to_use.append(col)
                    
            # Add any additional columns that weren't in the original order
            for col in sorted(all_columns):
                if col not in columns_to_use:
                    columns_to_use.append(col)
        else:
            # If we don't have original order, use alphabetical (after our important columns)
            columns_to_use.extend(sorted([col for col in all_columns 
                                        if col not in ['FILEPATH', 'mstudyid', 'filedate']]))
        
        # Ensure all dataframes have the same columns
        for i in range(len(all_data)):
            # Add missing columns
            for col in columns_to_use:
                if col not in all_data[i].columns:
                    all_data[i][col] = None
            
            # Reorder columns to match our desired order
            all_data[i] = all_data[i][columns_to_use]
        
        combined_data = pd.concat(all_data, ignore_index=True)
        combined_data.to_csv(os.path.join(output_dir, "combined_data.csv"), index=False)
        print(f"Combined data saved with {len(combined_data)} rows")
        
        # Print the columns to verify the order
        print("Columns in combined data:")
        print(combined_data.columns.tolist())
    else:
        combined_data = pd.DataFrame()
        print("No sample data was successfully processed")
    
    # Save summary
    with open(os.path.join(output_dir, "summary.txt"), 'w') as f:
        f.write(f"Total files processed: {total_files}\n")
        f.write(f"Successful: {success_count}\n")
        f.write(f"Errors: {error_count}\n")
        f.write(f"Properties rows: {len(combined_properties)}\n")
        f.write(f"Data rows: {len(combined_data)}\n")
    
    return combined_properties, combined_data

# # Example usage
# if __name__ == "__main__":
#     data_folder = "UPAS DATA"  # Update this path to your actual data folder
#     output_dir = "processed_data"
    
#     print(f"Starting processing of UPAS data from {data_folder}")
#     properties_df, data_df = process_upas_files(data_folder, output_dir)
#     print(f"Processing complete. Results saved to {output_dir}")
    
#     # Open the output folder automatically
#     open_output_folder(output_dir)

Starting processing of UPAS data from UPAS DATA


Processing files: 100%|██████████| 345/345 [00:25<00:00, 13.77file/s] , Success: 345, Errors: 0


Combined properties saved with 25875 rows
Combined data saved with 1901835 rows
Columns in combined data:
['FILEPATH', 'mstudyid', 'filedate', 'SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal', 'PumpingFlowRate', 'OverallFlowRate', 'SampledVolume', 'FilterDP', 'BatteryCharge', 'AtmoT', 'AtmoP', 'AtmoRH', 'AtmoDensity', 'AtmoAlt', 'GPSQual', 'GPSlat', 'GPSlon', 'GPSalt', 'GPSsat', 'GPSspeed', 'GPShDOP', 'AccelX', 'AccelXVar', 'AccelXMin', 'AccelXMax', 'AccelY', 'AccelYVar', 'AccelYMin', 'AccelYMax', 'AccelZ', 'AccelZVar', 'AccelZMin', 'AccelZMax', 'RotX', 'RotXVar', 'RotXMin', 'RotXMax', 'RotY', 'RotYVar', 'RotYMin', 'RotYMax', 'RotZ', 'RotZVar', 'RotZMin', 'RotZMax', 'AccelComplianceCnt', 'AccelComplianceHrs', 'Xup', 'XDown', 'Yup', 'Ydown', 'Zup', 'Zdown', 'StepCount', 'LUX', 'UVindex', 'HighVisRaw', 'LowVisRaw', 'IRRaw', 'UVRaw', 'PMMeasCnt', 'PM1MC', 'PM1MCVar', 'PM2_5MC', 'PM2_5MCVar', 'PM4MC', 'PM4MCVar', 'PM10MC', 'PM10MCVar', 'PM0_5NC', 'PM0_5NCVar', 'PM1NC

In [ ]:
# Example usage
data_folder = "UPAS DATA"  # Update this path to your actual data folder
output_dir = "processed_data"

print(f"Starting processing of UPAS data from {data_folder}")
properties_df, data_df = process_upas_files(data_folder, output_dir)
print(f"Processing complete. Results saved to {output_dir}")

# Open the output folder automatically
open_output_folder(output_dir)

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
data_df.sample(10)

,FILEPATH,FILENAME,AccelComplianceCnt,AccelComplianceHrs,AccelT,AccelX,AccelXMax,AccelXMin,AccelXVar,AccelY,AccelYMax,AccelYMin,AccelYVar,AccelZ,AccelZMax,AccelZMin,AccelZVar,AtmoAlt,AtmoDensity,AtmoP,AtmoRH,AtmoT,BCS1,BCS2,BC_NPG,BFGenergy,BattVolt,BatteryCharge,CO2,DateTimeLocal,DateTimeUTC,Dead,FLOWCTL,FdpT,FilterDP,GPSQual,GPSRT,GPSalt,GPShDOP,GPSlat,GPSlon,GPSsat,GPSspeed,HighVisRaw,IRRaw,LUX,LowVisRaw,MFSVout,MassFlow,NOXRaw,OverallFlowRate,P,PCB1T,PCB2P,PCB2T,PM0_5NC,PM0_5NCVar,PM10MC,PM10MCVar,PM10NC,PM10NCVar,PM1MC,PM1MCVar,PM1NC,PM1NCVar,PM2_5MC,PM2_5MCVar,PM2_5NC,PM2_5NCVar,PM2_5SampledMass,PM2_5SamxmE M!rs,PM4MC,PM4MCVar,PM4NC,PM4NCVar,PMFanErrorCnt,PMFanSpeedWarn,PMLaserErrorCnt,PMMeasCnt,PMReadingErrorCnt,PMRea|éngAvrorSnt,PMtypicalParticleSize,PMtypicalParticleSizeVar,PT100R,PumpPow1,PumpPow2,PumpV,PumpingFlowRate,PumpsON,RotX,RotXMax,RotXMin,RotXVar,RotY,RotYMax,RotYMin,RotYVar,RotZ,RotZMax,RotZMin,RotZVar,SCDRH,SCDT,SD_DATAW,SD_HEADW,SampleTime,SampledVolume,StepCount,TPumpsOFF,TPumpsON,UVRaw,UVindex,UnixTime,UnixTimeMCU,VOCRaw,XDown,Xup,Ydown,Yup,Zdown,Zup,v3_3,v5
972526,UPAS DATA/20250212/PSP00010_LOG_2025-02-10T11_...,PSP00010_LOG_2025-02-10T11_35_14UTC_HHC325____...,4,3.74,35.54,366.5,368,365,1.4,-948.6,-943,-951,3.3,-85.4,-75,-96,45.3,390.2,1.0753,967.25,33.89,37.56,1,1,1,43687,3.88,078,,2025-02-10T21:35:01,2025-02-10T21:35:01,0,0.353000,38.98,449.40,,0.000000,,,,,,,-14.1,1.9,0.1,0.0,1.361250,1.07885,,0.200,None,37.93,967.6,38.07,,,,,,,,,,,,,,,,None,,,,,,,,,,None,,,114.60,463,0,11.041,0.999,1,159.7,2808,-1793,1010738.4,-528.2,-131,-857,23234.5,37.5,647,-332,38534.2,,,0.015000,0.034000,9:59:39,119.831,0,23.900999,6.101000,32.1,0.29,1739223301,1739223302,,0.0,0.0,0.0,100.0,0.0,0.0,3.34,4.98
1261826,UPAS DATA/20250215/PSP00303_LOG_2025-02-13T07_...,PSP00303_LOG_2025-02-13T07_53_44UTC_HHC196____...,0,3.26,31.19,48.0,48,47,0.0,98.2,99,98,0.2,-973.8,-973,-976,0.6,362.8,1.0900,970.43,42.29,34.25,1,1,1,38387,3.78,066,,2025-02-14T01:34:30,2025-02-14T01:34:30,0,0.363000,35.13,186.70,,0.000000,,,,,,,-5.6,1.6,0.1,0.7,1.317000,1.08911,,0.200,None,34.08,971.1,34.22,,,,,,,,,,,,,,,,None,,,,,,,,,,None,,,113.32,593,0,9.208,0.999,1,-15536.4,175,-83221,985339520.0,-598.2,-490,-647,971.5,-244.3,-210,-332,647.5,,,0.016000,0.034000,17:40:39,211.918,0,23.885000,6.120000,25.5,0.27,1739496870,1739496873,,0.0,0.0,0.0,0.0,0.0,100.0,3.34,5.00
153927,UPAS DATA/20250105/PSP00010_LOG_2025-01-03T07_...,PSP00010_LOG_2025-01-03T07_39_24UTC_HHC042____...,0,6.53,28.76,410.3,411,410,0.2,207.1,208,207,0.1,-857.2,-856,-858,0.1,323.2,1.1096,975.03,35.73,31.06,1,1,1,37126,3.72,063,,2025-01-04T04:23:30,2025-01-04T04:23:30,0,0.421000,32.22,355.15,,0.000000,,,,,,,-9.7,1.7,0.1,-0.1,1.309375,1.10470,,0.200,None,32.00,975.3,32.22,430.54,41.34,84.16,0.48,512.08,47.56,64.16,0.75,501.93,49.22,75.19,0.65,510.38,47.77,20.9570,None,81.16,0.53,512.08,47.56,0,0,0,28,2,None,0.56,0.00,112.08,545,0,10.259,0.999,1,178.6,227,122,842.5,-709.2,-665,-761,477.8,-127.2,-96,-148,153.5,,,0.020000,0.038000,20:43:58,248.353,0,23.907000,6.094000,46.5,0.38,1735964610,1735964612,,0.0,0.0,0.0,0.0,0.0,100.0,3.35,4.98
1669093,UPAS DATA/20250303/PSP00299_LOG_2025-02-27T11_...,PSP00299_LOG_2025-02-27T11_33_41UTC_HHC286____...,0,13.68,34.92,953.2,954,953,-0.1,3.3,4,3,0.2,323.6,324,323,0.2,309.6,1.0964,976.61,43.68,34.34,1,1,1,33496,3.59,054,,2025-02-28T10:55:30,2025-02-28T10:55:30,0,0.437000,35.08,188.80,,0.000000,,,,,,,-8.4,1.7,0.1,-0.1,1.362875,1.09832,,0.200,None,35.10,977.3,34.20,291.39,5.42,47.97,0.42,335.48,8.91,42.03,0.14,332.77,8.10,46.03,0.26,335.06,8.78,19.0842,None,47.32,0.36,335.48,8.91,0,0,0,30,0,None,0.51,0.00,113.35,587,0,9.326,0.999,1,-10120.2,245,-54775,427203072.0,-365.9,-227,-437,2261.0,-211.4,-183,-253,267.3,,,0.020000,0.037000,23:21:41,279.990,0,23.882999,6.121000,-39.9,0.33,1740740130,1740740133,,100.0,0.0,0.0,0.0,0.0,0.0,3.34,4.99
343592,UPAS DATA/20250108/PSP00024_LOG_2025-01-06T11_...,PSP00024_LOG_2025-01-06T11_17_22UTC_HHC105____...,0,8.81,29.86,80.5,81,80,

In [6]:
data_df.columns

Index(['FILEPATH', 'FILENAME', 'AccelComplianceCnt', 'AccelComplianceHrs',
       'AccelT', 'AccelX', 'AccelXMax', 'AccelXMin', 'AccelXVar', 'AccelY',
       ...
       'UnixTimeMCU', 'VOCRaw', 'XDown', 'Xup', 'Ydown', 'Yup', 'Zdown', 'Zup',
       'v3_3', 'v5'],
      dtype='object', length=123)

In [9]:
data_df.drop(columns=['FILENAME'], inplace=True)

In [10]:
column_list = data_df.columns.tolist()
# column_list

In [11]:
print(column_list)


['FILEPATH', 'AccelComplianceCnt', 'AccelComplianceHrs', 'AccelT', 'AccelX', 'AccelXMax', 'AccelXMin', 'AccelXVar', 'AccelY', 'AccelYMax', 'AccelYMin', 'AccelYVar', 'AccelZ', 'AccelZMax', 'AccelZMin', 'AccelZVar', 'AtmoAlt', 'AtmoDensity', 'AtmoP', 'AtmoRH', 'AtmoT', 'BCS1', 'BCS2', 'BC_NPG', 'BFGenergy', 'BattVolt', 'BatteryCharge', 'CO2', 'DateTimeLocal', 'DateTimeUTC', 'Dead', 'FLOWCTL', 'FdpT', 'FilterDP', 'GPSQual', 'GPSRT', 'GPSalt', 'GPShDOP', 'GPSlat', 'GPSlon', 'GPSsat', 'GPSspeed', 'HighVisRaw', 'IRRaw', 'LUX', 'LowVisRaw', 'MFSVout', 'MassFlow', 'NOXRaw', 'OverallFlowRate', 'P', 'PCB1T', 'PCB2P', 'PCB2T', 'PM0_5NC', 'PM0_5NCVar', 'PM10MC', 'PM10MCVar', 'PM10NC', 'PM10NCVar', 'PM1MC', 'PM1MCVar', 'PM1NC', 'PM1NCVar', 'PM2_5MC', 'PM2_5MCVar', 'PM2_5NC', 'PM2_5NCVar', 'PM2_5SampledMass', 'PM2_5SamxmE M!rs', 'PM4MC', 'PM4MCVar', 'PM4NC', 'PM4NCVar', 'PMFanErrorCnt', 'PMFanSpeedWarn', 'PMLaserErrorCnt', 'PMMeasCnt', 'PMReadingErrorCnt', 'PMRea|éngAvrorSnt', 'PMtypicalParticleSize